In [ ]:
!pip install -q torch transformers datasets evaluate accelerate

In [ ]:
%%writefile engine_core.py
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class SentimentAnalyzer:
    def __init__(self, items, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        if not items or any(not item["text"].strip() for item in items):
            raise ValueError("Items list cannot be empty or contain empty text fields")
        
        self.items = items
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()

    def analyze_batch(self, texts):
        if not texts:
                    return []
        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
        return probs.tolist()

    def classify(self, query, top_k=2):
        if not query.strip():
            return []
        scores = self.analyze_batch([query])[0]
        results = []
        labels = ["NEGATIVE", "POSITIVE"]
        for idx, score in enumerate(scores):
            results.append({"label": labels[idx], "score": float(score)})
        
        results = sorted(results, key=lambda x: x["score"], reverse=True)
        return results[:top_k]

def display_summary(results):
    if not results:
        print("No classification results found.")
        return
    for res in results:
        print(f"Label: {res['label']} | Confidence: {res['score']:.4f}")

In [ ]:
%%writefile nlp_pipeline.py
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('punkt', quiet=True)

samples = [
    "Machine learning powers modern recommendation systems.",
    "The quick brown fox jumps over the lazy dog!",
    "API endpoint responded with status code 500.",
    "ڈویلپمنٹ کے دوران کوئ پرابلم نہیں آئی۔",
    "v3.5-alpha release candidate #42"
]

for text in samples:
    tokens = word_tokenize(text)
    sentences = sent_tokenize(text)
    print(repr(text), "| Sentences:", len(sentences), "| Tokens:", len(tokens))

assert len(word_tokenize("")) == 0
print("Tokenization checks passed successfully.")

In [ ]:
%%writefile interactive_demo.py
from engine_core import SentimentAnalyzer

KB_DATA = [
    {"id": "kb:refund",
     "question": "Customer ko product return karne par refund kitne din mein milta hai?",
     "answer": "Inspection complete hone ke baad 3 se 5 working days mein amount bank account ya card mein transfer ho jati hai.",
     "source": "policies.txt"},

    {"id": "kb:delivery",
     "question": "Order dispatch hone ke baad delivery mein kitna time lagta hai?",
     "answer": "Standard delivery 2 se 4 business days leti hai, jab ke express delivery 24 hours mein deliver ho jati hai.",
     "source": "policies.txt"},

    {"id": "kb:warranty",
     "question": "Electronics items par brand warranty kitne arsay ki milti hai?",
     "answer": "Tamam branded appliances aur gadgets par 1 year ki official manufacturer warranty faraham ki jati hai.",
     "source": "policies.txt"},

    {"id": "kb:discount",
     "question": "First-time buyer ke liye koi promotional discount code available hai?",
     "answer": "G han, aap checkout ke waqt 'WELCOME10' use karke apni pehli purchase par 10% instant discount hasil kar sakte hain.
     "source": "policies.txt"},

    {"id": "kb:support",
     "question": "Customer support helpline kis time tak active rehti hai?",
     "answer": "Hamari support team subah 9 baje se raat 10 baje tak live chat aur phone calls par dastyab hoti hai.",
     "source": "policies.txt"}
]

def initialize_assistant():
    items = []
    for record in KB_DATA:
        payload = dict(record)
        payload["text"] = record["question"] + " " + record["answer"]
        items.append(payload)
    return SentimentAnalyzer(items)

def process_query(analyzer, user_input):
    if not user_input.strip():
        return "Please provide a valid query string."
    for entry in KB_DATA:
        if user_input.strip().casefold() == entry["question"].casefold():
            return entry["answer"] + " [ref: " + entry["source"] + "]"
    
    evals = analyzer.classify(user_input)
    if not evals:
        return "No relevant sentiment or match found. Contact support."
    
    response_lines = ["Text Sentiment & Classification Insights:"]
    for item in evals:
        response_lines.append(f"- Category: {item['label']} (Confidence Score: {item['score']:.4f})")
    return "\n".join(response_lines)

if __name__ == "__main__":
    import sys
    print("AI Knowledge Assistant initialized. Type 'quit' to exit.")
    analyzer = initialize_assistant()
    while True:
        try:
            query = input("User: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "quit":
            break
        result = process_query(analyzer, query)
        print("Assistant:", result)

In [ ]:
!python nlp_pipeline.py

In [ ]:
%run interactive_demo.py